In [1]:
# 검색해서 평가 및 테스트 하는 작업 공간.

In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

In [3]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

In [4]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

target_user = '김부장' #input('이름을 입력해주세요: ')
CHROMA_PATH = f"./data/{target_user}/chroma_db"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = chromadb.PersistentClient(path=CHROMA_PATH)


vectorstore = Chroma(
    client=client,
    embedding_function=embeddings
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/var/folders/8n/knn0qxl17sq7yn7ywq1gjgtm0000gn/T/ipykernel_27683/1095374796.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [5]:
from datetime import date

today = date.today().strftime("%Y-%m-%d")
current_year = date.today().year


document_content_description = f"""
{target_user}의 일상과 감정을 기록한 개인 일기 모음.
"""

metadata_field_info = [
    AttributeInfo(
        name="date",
        description="일기 날짜. '언제야?'와 같은 질문에는 특정 기간으로 제한하기보다 전체 기간을 탐색한 뒤 결과를 날짜순으로 정렬하도록 쿼리를 작성하세요.",
        type="string",
    ),
    AttributeInfo(
        name="emotion",
        description="일기의 주요 감정. 반드시 다음 중 하나: 기쁨, 놀라움, 두려움, 분노, 불쾌함, 설렘, 슬픔, 평범함.",
        type="string",
    )
]


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    search_kwargs={"k": 10},
    verbose=True
)

In [ ]:
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Optional

"""
query를 바로 사용하여 검색을 진행할 경우, 검색 성능 향상을 위해 query에 주는 정보를 수정
1. 날짜 정보를 명확히 제공 - 작년, 최근 등과 같은 입력에 대해 대응할 수 있도록 한다.
2. 감정 표현이 애매한 경우 예를 들어 행복이라는 단어는 감정 분류에 존재하지 않는데, 이를 해결하기 위해 감정과 유사한 감정으로 매칭될 수 있도록 확장.
3. 텍스트에서 주요한 단어를 뽑아내는 과정 추가 (쿼리에서 중요하게 생각하는 인물 등...)
"""



# 검색 의도 분석 - 데이터 구조 정의
class SearchIntent(BaseModel):
    refined_query: str = Field(description="검색에 사용할 핵심 사건 키워드")
    start_date: Optional[str] = Field(description="필터링 시작 날짜 (YYYY-MM-DD)")
    end_date: Optional[str] = Field(description="필터링 종료 날짜 (YYYY-MM-DD)")
    target_emotions: List[str] = Field(description="확장된 감정 리스트 (유사 감정 포함)")

# 쿼리 분석
parser = JsonOutputParser(pydantic_object=SearchIntent)

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 {target_user}의 텍스트 전문가입니다. 오늘 날짜는 {today}입니다.
사용자의 질문을 분석하여 최적의 검색 파라미터를 JSON으로 생성하세요.

[지침]
- 감정 확장: 사용자가 언급한 감정과 유사한 감정을 리스트에 포함하세요.
  (예: '행복' -> ['기쁨', '설렘'], '짜증' -> ['불쾌함', '분노'])
- 감정 후보: 기쁨, 놀라움, 두려움, 분노, 불쾌함, 설렘, 슬픔, 평범함

{format_instructions}"""),
    ("user", "{query}")
])

analysis_prompt = analysis_prompt.partial(
    target_user=target_user,
    today=today,
    format_instructions=parser.get_format_instructions()
)

analyzer_chain = analysis_prompt | llm | parser

# 키워드 + 감정을 합쳐서 임베딩 검색 쿼리로 사용
def smart_diary_retrieve(query, vectorstore, k=10):
    # 쿼리의 의도 분석
    intent = analyzer_chain.invoke({"query": query})
    print(f"\n[분석 결과] 키워드: {intent['refined_query']}, 감정: {intent['target_emotions']}")

    # 키워드 + 감정을 결합 - 풍부한 검색 쿼리 생성
    emotions_str = " ".join(intent['target_emotions'])
    enriched_query = f"{intent['refined_query']} {emotions_str}".strip()
    print(f"[검색 쿼리] {enriched_query}")

    # 벡터 유사도 검색
    docs = vectorstore.similarity_search(enriched_query, k=k)

    return docs

/Users/siyoung/Archive-of-Feelings/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# test case1

test_query = "상무님께 칭찬받았던 날은 언제야?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


In [11]:
# test case2

test_query = "동기였던 박 부장이 짐을 쌌다는 소식을 들은 날은?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 박 부장이 짐을 쌌다는 소식, 감정: ['슬픔', '불쾌함']
[검색 쿼리] 박 부장이 짐을 쌌다는 소식 슬픔 불쾌함

최종 검색 결과 (10건):
[2026-03-16] (두려움) 동기였던 박 부장이 오늘 짐을 쌌다. 특별한 예고도 없었는데... 그 뒷모습을 보니 남 일 같지 않아 가슴이 철렁했다. 나는 언제까지 이 책상을 지킬 수 있을까. 퇴근길 지하철 차...
[2026-02-27] (슬픔) 친하게 지내던 김 부장이 명예퇴직 신청을 했다는 소식을 들었다. 점심을 같이 먹는데 그 형님의 눈가가 촉촉해 보여서 나까지 마음이 아팠다. 30년 가까이 바친 청춘의 끝이 서류 한...
[2026-03-11] (분노) 위에서 또 무리한 지시가 내려왔다. 현장 상황은 전혀 모르면서 무조건 일주일 안에 끝내라니. 팀원들 얼굴 볼 면목이 없다. 중간에 끼어 있는 부장 자리가 오늘따라 너무나 고달프고 ...
[2026-01-16] (분노) 금요일. 회의 시간에 내 아이디어를 가로채서 본인 것인 양 발표하는 옆 부서 부장 때문에 화가 폭발할 뻔했다. 인간관계의 밑바닥을 본 기분이다. 어떻게 저렇게 뻔뻔할 수 있는지, ...
[2025-12-26] (분노) 금요일. 회의 시간에 말도 안 되는 논리로 나를 공격하는 후배 부장 때문에 화가 났다. 경쟁 관계라지만 지켜야 할 선이 있는데... 비겁한 행동에 실망감이 크고 화가 풀리지 않는다...
[2026-02-06] (분노) 금요일. 또 야근이다. 위에서 내린 지시가 하필 퇴근 직전에... 다들 눈치 보느라 못 나가는 상황이 정말 싫다. 부장인 내가 먼저 총대를 메고 퇴근하려 했지만 분위기상 실패했다....
[2025-12-17] (슬픔) 수요일. 친했던 동료의 부친상 소식을 들었다. 장례식장에 다녀오는데 마음이 너무 무겁고 슬프다. 부모님과의 이별은 언제나 갑작스럽고 아픈 법이다. 살아계신 부모님께 더 잘해드려야겠...
[2026-02-18] (기쁨) 수요일. 오랫동안 준비한 계약이 드디어 성사됐다. 조건

In [12]:
# test case3

test_query = "가장 최근에 아들이랑 산책하면서 든든함을 느꼈던 날은 언제지?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 아들과 산책, 감정: ['안도', '안정감', '기쁨']
[검색 쿼리] 아들과 산책 안도 안정감 기쁨

최종 검색 결과 (10건):
[2026-03-15] (기쁨) 일요일. 아들이 웬일로 먼저 산책을 가자고 했다. 중학교 이후로 처음인 것 같다. 같이 걷는 내내 대화는 별로 없었지만, 녀석의 어깨가 나만큼 넓어진 걸 보니 든든하고 뿌듯했다. ...
[2025-12-21] (기쁨) 일요일. 오랜만에 아들과 캐치볼을 했다. 아들의 던지는 공에 힘이 실린 걸 느끼며 참 많이 컸구나 싶어 대견했다. 운동하며 땀 흘리고 대화도 나누니 관계가 한결 가까워진 기분이라 ...
[2026-01-21] (슬픔) 수요일. 비 오는 퇴근길, 라디오에서 흘러나오는 슬픈 노래에 나도 모르게 눈물이 났다. 가장으로서의 책임감, 회사원으로서의 중압감... 누군가에게 털어놓고 싶은데 마땅한 사람이 없...
[2026-02-11] (슬픔) 수요일. 퇴근길에 아버지가 좋아하시던 단팥빵을 샀다. 이미 돌아가신 지 오래지만, 가끔 이렇게 특정 음식을 보면 아버지가 사무치게 그립다. 살아계실 때 더 잘해드릴걸, 후회는 항상...
[2026-03-13] (슬픔) 고향 계신 어머니께서 전화를 하셨는데, 목소리가 예전보다 훨씬 작아지셨다. 바쁘다는 핑계로 설날 이후 한 번도 못 찾아뵀는데 마음이 무겁다. 효도하고 싶어도 시간이 기다려주지 않는...
[2026-01-02] (분노) 금요일. 신년 초부터 터무니없는 목표치를 들고 온 임원 때문에 회의장 분위기가 싸늘했다. 현실성 없는 계획에 다들 기가 차서 말이 안 나온다. 책임은 실무진이 지고 생색은 본인이 ...
[2026-02-20] (놀라움) 아들놈이 이번 달 용돈을 아껴서 내 생일 선물을 미리 사 왔다고 한다. 무뚝뚝한 녀석이라 기대도 안 했는데, 정성껏 쓴 짧은 편지까지... 녀석의 성장에 정말 놀랐고 가슴 벅찬 감...
[2025-12-17] (슬픔) 수요일. 친했던 동료의 부친상 소식을 들었다. 장례식장에 다녀오는데 마

In [ ]:
# test case4
test_query = "작년 12월에 눈이 많이 와서 놀랐던 적이 있어?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 2025년 12월 눈, 감정: ['놀라움', '기쁨', '설렘']
[검색 쿼리] 2025년 12월 눈 놀라움 기쁨 설렘

최종 검색 결과 (10건):
[2025-12-18] (놀라움) 목요일. 눈이 아주 많이 내렸다. 온 세상이 하얗게 변한 풍경이 정말 놀랍고 아름답다. 출근길은 힘들었지만 잠시 동심으로 돌아간 기분이었다. 눈 덮인 나뭇가지들이 비현실적으로 예뻐...
[2026-02-20] (놀라움) 아들놈이 이번 달 용돈을 아껴서 내 생일 선물을 미리 사 왔다고 한다. 무뚝뚝한 녀석이라 기대도 안 했는데, 정성껏 쓴 짧은 편지까지... 녀석의 성장에 정말 놀랐고 가슴 벅찬 감...
[2026-01-01] (설렘) 새해 첫날. 가족과 함께 해돋이를 보러 갔다. 떠오르는 태양을 보며 올 한 해 우리 가족의 건강과 행복을 빌었다. 새로운 시작이 주는 설렘과 희망으로 가슴이 벅차오른다. 올해는 정...
[2026-01-14] (슬픔) 수요일. 거울 속의 내가 낯설게 느껴지는 날이다. 언제 이렇게 늙어버렸나. 주름진 얼굴과 처진 어깨를 보니 세월의 무상함이 느껴져 가슴이 아릿하다. 다시 돌아갈 수 없는 젊음이 그...
[2025-12-11] (놀라움) 목요일. 10년 전 기록했던 일기장을 발견했다. 그때의 고민이 지금 보니 별것 아니더라. 시간이 지나면 다 해결된다는 걸 다시 깨달아 놀라웠다. 과거의 나를 만나고 온 기분이라 신...
[2025-12-31] (슬픔) 올해의 마지막 날. 한 해를 돌아보니 아쉬움이 많이 남는다. 더 잘할 수 있었던 일들, 상처 준 말들... 세월은 이렇게 또 한 페이지를 넘기는데 나는 여전히 제자리인 것 같아 쓸...
[2026-02-04] (슬픔) 수요일. 입춘이라는데 날씨는 여전히 춥다. 마음속에도 아직 봄은 오지 않은 것 같다. 회사 생활 25년, 얻은 것과 잃은 것을 따져보니 공허함만 남는 기분이다. 오늘은 유독 술이 ...
[2026-01-15] (놀라움) 목요일. 10년 전에 도와줬던 협력업체 직원이 

In [14]:
# test case5
test_query = "딸아이가 첫 월급을 타고 내 영양제를 사 왔던 감동적인 날은?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 딸아이 첫 월급 영양제, 감정: ['기쁨', '설렘']
[검색 쿼리] 딸아이 첫 월급 영양제 기쁨 설렘

최종 검색 결과 (10건):
[2026-03-08] (기쁨) 딸아이가 첫 월급을 탔다며 내 영양제를 사 왔다. 아빠 건강 챙겨주는 건 역시 딸밖에 없다. 영양제를 먹기도 전부터 힘이 불끈 솟는 기분이다. 그동안 고생하며 회사 다닌 보람을 이...
[2026-02-20] (놀라움) 아들놈이 이번 달 용돈을 아껴서 내 생일 선물을 미리 사 왔다고 한다. 무뚝뚝한 녀석이라 기대도 안 했는데, 정성껏 쓴 짧은 편지까지... 녀석의 성장에 정말 놀랐고 가슴 벅찬 감...
[2026-01-01] (설렘) 새해 첫날. 가족과 함께 해돋이를 보러 갔다. 떠오르는 태양을 보며 올 한 해 우리 가족의 건강과 행복을 빌었다. 새로운 시작이 주는 설렘과 희망으로 가슴이 벅차오른다. 올해는 정...
[2026-02-10] (기쁨) 화요일. 드디어 다이어트 시작 후 3kg 감량에 성공했다! 몸이 한결 가볍고 아침에 일어날 때 개운하다. 노력하면 아직 바뀔 수 있다는 자신감이 생겨서 기분이 정말 좋다. 이 기세...
[2026-01-12] (불쾌함) 월요일. 점심 식사 중에 국이 옷에 튀었다. 흰 와이셔츠인데... 하루 종일 얼룩진 옷을 입고 있으려니 신경 쓰이고 기분이 불쾌하다. 업무 미팅도 있었는데 자신감이 떨어져서 힘들었...
[2026-01-13] (기쁨) 화요일. 드디어 허리 통증이 사라졌다! 꾸준히 치료받고 스트레칭한 보람이 있다. 마음껏 걸을 수 있고 앉아 있을 때 통증이 없으니 세상이 달라 보인다. 건강이 역시 최고의 자산이라...
[2026-02-14] (설렘) 밸런타인데이라며 딸아이가 직접 만든 초콜릿을 줬다. 달콤한 향기가 온 집안에 퍼지니 기분까지 달달해진다. 다음 달 화이트데이 때는 딸이 좋아하는 맛있는 케이크를 사다 줘야겠다고 미...
[2026-01-29] (놀라움) 목요일. 출근길에 길 잃은 강아지를 발견해서 주인에게 찾아줬다. 

In [15]:
# test case6
test_query = "상김 차장이 때문에 최근에 짜증났던 일은 언제야?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 상김 차장 짜증, 감정: ['불쾌함', '분노']
[검색 쿼리] 상김 차장 짜증 불쾌함 분노

최종 검색 결과 (10건):
[2026-03-18] (분노) 김 차장이 또 선을 넘는다. 보고서 수정을 지시했더니 '요즘 방식은 이게 맞다'며 가르치려 든다. 나도 나름 공부하고 확인한 건데 무시당한 기분이다. 술 한잔 생각나지만 당 수치 ...
[2026-02-16] (두려움) 건강검진 결과표를 다시 들여다봤다. 간 수치가 높게 나왔는데, 술 때문이겠지. 겁이 난다. 아직 할 일이 많은데 몸이 망가지면 어쩌나. 오늘부터는 정말 식단 조절하고 술도 줄여야겠...
[2026-01-23] (분노) 금요일. 주차장에서 내 차를 긁고 도망간 놈을 잡았다. 미안하다는 말 한마디 없이 보험 처리하라고 적반하장으로 나오는데, 정말 양심 없는 인간들이 많구나 싶어 분노가 치밀었다. 법...
[2026-02-25] (기쁨) 수요일. 보고서 통과! 상무님께 드물게 칭찬을 받았다. 밤새워 준비한 보람이 있다. 팀원들한테 고맙다고 메일 한 통씩 돌렸다. 성과가 날 때의 이 짜릿함 때문에 아직도 회사 생활을...
[2026-02-13] (분노) 금요일. 회의 시간에 내 의견을 대놓고 묵살하는 신입 팀원 때문에 얼굴이 화끈거렸다. 논리도 부족하면서 목소리만 크니... 세대 차이를 넘어서 기본 예의의 문제인 것 같다. 화를 ...
[2026-02-06] (분노) 금요일. 또 야근이다. 위에서 내린 지시가 하필 퇴근 직전에... 다들 눈치 보느라 못 나가는 상황이 정말 싫다. 부장인 내가 먼저 총대를 메고 퇴근하려 했지만 분위기상 실패했다....
[2025-12-12] (분노) 금요일. 회의 시간에 내 실수를 대놓고 비웃는 후배 때문에 모멸감을 느꼈다. 화가 나지만 참아야 하는 내 위치가 싫다. 무시당하지 않으려면 실력을 더 키워야 하나, 자괴감이 들고 ...
[2026-02-27] (슬픔) 친하게 지내던 김 부장이 명예퇴직 신청을 했다는 소식을 들었다. 점심을 같이 먹

In [16]:
# test case7
test_query = "혈압이 높게 나와서 가족들이 걱정했던 날은?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 혈압 걱정, 감정: ['두려움', '불쾌함', '슬픔']
[검색 쿼리] 혈압 걱정 두려움 불쾌함 슬픔

최종 검색 결과 (10건):
[2026-03-05] (두려움) 오늘 혈압을 쟀는데 또 높게 나왔다. 약을 먹어야 할 수준이라는데, 건강을 너무 과신했나 보다. 내가 쓰러지면 남겨진 가족들은 어떡하나 하는 불안감이 엄습한다. 당분간 술을 끊고 ...
[2025-12-23] (불쾌함) 화요일. 감기 기운이 있어서 하루 종일 몸이 으슬으슬하다. 억지로 업무를 보려니 집중력도 떨어지고 자꾸 짜증이 난다. 건강이 제일인데... 아픈 내 자신이 원망스럽고 기분도 축 처...
[2026-02-24] (불쾌함) 날씨가 갑자기 추워졌다. 출근길에 마스크를 안 챙겨서 목이 칼칼하다. 점심 먹으러 간 식당은 또 왜 그렇게 불친절한지. 별것 아닌 일에 기분이 상하는 걸 보니 마음의 여유가 많이 ...
[2026-02-16] (두려움) 건강검진 결과표를 다시 들여다봤다. 간 수치가 높게 나왔는데, 술 때문이겠지. 겁이 난다. 아직 할 일이 많은데 몸이 망가지면 어쩌나. 오늘부터는 정말 식단 조절하고 술도 줄여야겠...
[2025-12-15] (불쾌함) 월요일. 출근길 버스에서 옆 사람이 너무 밀쳐서 짜증이 났다. 사과도 없이 인상만 쓰고... 아침부터 기분을 상하니 하루 종일 마음이 꼬여 있다. 사소한 배려가 없는 사람들이 너무...
[2026-03-17] (불쾌함) 비가 내려서 그런지 무릎이 유독 쑤신다. 출근길 지하철에서 누가 내 발을 꽉 밟고 사과도 없이 가버렸다. 아침부터 기분을 잡치니 하루 종일 업무 집중이 안 된다. 나이를 먹으니 작...
[2026-01-09] (분노) 금요일. 위에서 시킨 불합리한 업무 때문에 팀원들이 고생하는 걸 보니 속이 상하고 화가 난다. 현장의 목소리는 무시하고 수치만 따지는 상부의 태도가 정말 환멸 난다. 부장으로서 막...
[2026-01-21] (슬픔) 수요일. 비 오는 퇴근길, 라디오에서 흘러나오는 슬픈 노래에 

In [17]:
# test case8
test_query = "내 아이디어를 가로채서 본인 것인 양 발표한 사람은 누구야?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 아이디어 가로채기 발표, 감정: ['분노', '불쾌함']
[검색 쿼리] 아이디어 가로채기 발표 분노 불쾌함

최종 검색 결과 (10건):
[2026-01-16] (분노) 금요일. 회의 시간에 내 아이디어를 가로채서 본인 것인 양 발표하는 옆 부서 부장 때문에 화가 폭발할 뻔했다. 인간관계의 밑바닥을 본 기분이다. 어떻게 저렇게 뻔뻔할 수 있는지, ...
[2026-02-25] (기쁨) 수요일. 보고서 통과! 상무님께 드물게 칭찬을 받았다. 밤새워 준비한 보람이 있다. 팀원들한테 고맙다고 메일 한 통씩 돌렸다. 성과가 날 때의 이 짜릿함 때문에 아직도 회사 생활을...
[2026-02-18] (기쁨) 수요일. 오랫동안 준비한 계약이 드디어 성사됐다. 조건 조율이 힘들어서 포기할까도 생각했지만 끝까지 버틴 보람이 있다. 팀원들과 가볍게 맥주 한잔하며 자축했다. 오늘만큼은 정말 행...
[2026-01-27] (기쁨) 화요일. 드디어 막혔던 프로젝트 아이디어가 떠올랐다! 화장실에서 볼일 보다 갑자기 스친 생각인데, 정말 완벽하다. 역시 고민은 멈추지 말아야 하나 보다. 해결의 실마리를 찾으니 온...
[2026-02-26] (두려움) 뉴스에서 대기업 구조조정 이야기가 연일 나온다. 우리 회사도 조만간 칼바람이 불 거라는 소문이 파다하다. 애들은 아직 대학도 안 마쳤는데... 밤늦도록 거실 불을 끄지 못하고 한숨...
[2026-01-09] (분노) 금요일. 위에서 시킨 불합리한 업무 때문에 팀원들이 고생하는 걸 보니 속이 상하고 화가 난다. 현장의 목소리는 무시하고 수치만 따지는 상부의 태도가 정말 환멸 난다. 부장으로서 막...
[2026-03-10] (평범함) 화요일. 점심으로 동태찌개를 먹었는데 시원해서 좋았다. 오후엔 회의 준비로 좀 바빴지만 퇴근은 제시간에 했다. 현관문을 열 때 강아지가 반겨주는 그 순간이 하루 중 가장 평화로운 ...
[2026-02-13] (분노) 금요일. 회의 시간에 내 의견을 대놓고 묵살하는 신입 팀원 때문

In [8]:
# test case9
test_query = "20년 전 신입사원 때 사수였던 최 이사님을 우연히 만난 날에 대해 기록한 날은 언제야?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 최 이사님 우연히 만난 날, 감정: ['놀라움', '설렘']
[검색 쿼리] 최 이사님 우연히 만난 날 놀라움 설렘


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



최종 검색 결과 (10건):
[2026-03-12] (놀라움) 세상에, 점심 먹으러 나갔다가 20년 전 신입사원 때 사수였던 최 이사님을 마주쳤다. 백발이 되셨지만 눈빛은 여전하시더라. 짧은 인사였지만 옛 생각이 나며 뭉클했다. 그 시절의 열...
[2025-12-11] (놀라움) 목요일. 10년 전 기록했던 일기장을 발견했다. 그때의 고민이 지금 보니 별것 아니더라. 시간이 지나면 다 해결된다는 걸 다시 깨달아 놀라웠다. 과거의 나를 만나고 온 기분이라 신...
[2025-12-15] (불쾌함) 월요일. 출근길 버스에서 옆 사람이 너무 밀쳐서 짜증이 났다. 사과도 없이 인상만 쓰고... 아침부터 기분을 상하니 하루 종일 마음이 꼬여 있다. 사소한 배려가 없는 사람들이 너무...
[2026-01-29] (놀라움) 목요일. 출근길에 길 잃은 강아지를 발견해서 주인에게 찾아줬다. 주인이 울면서 고맙다고 인사를 하는데, 내 작은 행동이 누군가에게 큰 기쁨이 될 수 있다는 게 놀라웠다. 마음 한구...
[2026-01-14] (슬픔) 수요일. 거울 속의 내가 낯설게 느껴지는 날이다. 언제 이렇게 늙어버렸나. 주름진 얼굴과 처진 어깨를 보니 세월의 무상함이 느껴져 가슴이 아릿하다. 다시 돌아갈 수 없는 젊음이 그...
[2025-12-18] (놀라움) 목요일. 눈이 아주 많이 내렸다. 온 세상이 하얗게 변한 풍경이 정말 놀랍고 아름답다. 출근길은 힘들었지만 잠시 동심으로 돌아간 기분이었다. 눈 덮인 나뭇가지들이 비현실적으로 예뻐...
[2026-02-20] (놀라움) 아들놈이 이번 달 용돈을 아껴서 내 생일 선물을 미리 사 왔다고 한다. 무뚝뚝한 녀석이라 기대도 안 했는데, 정성껏 쓴 짧은 편지까지... 녀석의 성장에 정말 놀랐고 가슴 벅찬 감...
[2026-01-05] (불쾌함) 월요일. 출근길에 어떤 무개념 운전자가 물웅덩이를 밟고 지나가서 옷이 다 젖었다. 사과도 없이 도망가는 뒷모습에 화가 머리끝까지 났고, 하루 종일 찝찝한 기분으로 업무를 봐야 했다...


In [21]:
# test case10
test_query = "최근에 퇴근하고 아내가 해준 어떤 음식을 먹고 편안함을 느꼈어?"
results = smart_diary_retrieve(test_query, vectorstore)

print(f"\n최종 검색 결과 ({len(results)}건):")
for doc in results:
    print(f"[{doc.metadata.get('date')}] ({doc.metadata.get('emotion')}) {doc.page_content[:100]}...")


[분석 결과] 키워드: 퇴근 후 아내가 해준 음식, 감정: ['평범함', '안정감', '편안함']
[검색 쿼리] 퇴근 후 아내가 해준 음식 평범함 안정감 편안함

최종 검색 결과 (10건):
[2026-02-11] (슬픔) 수요일. 퇴근길에 아버지가 좋아하시던 단팥빵을 샀다. 이미 돌아가신 지 오래지만, 가끔 이렇게 특정 음식을 보면 아버지가 사무치게 그립다. 살아계실 때 더 잘해드릴걸, 후회는 항상...
[2026-01-03] (설렘) 토요일. 아내와 함께 맛집 탐방을 다녀왔다. 유명한 곳이라 줄을 서서 기다렸지만 기다린 보람이 있는 맛이었다. 새로운 맛을 발견하고 아내와 즐거운 대화를 나누니 데이트하는 기분이 ...
[2026-01-17] (설렘) 토요일. 은퇴 후에 아내와 같이 살 전원주택 부지를 보러 다녀왔다. 조용한 동네와 깨끗한 공기가 벌써부터 마음을 설레게 한다. 마당에 어떤 꽃을 심을지 아내와 상의하며 행복한 상상...
[2026-03-04] (기쁨) 수요일. 오랜만에 일찍 퇴근해서 아내와 영화를 봤다. 연애할 때처럼 팝콘도 나눠 먹고 웃었다. 바쁘다는 핑계로 가장 소중한 사람을 너무 소홀히 대했던 건 아닌지 반성하게 된, 따뜻...
[2026-03-10] (평범함) 화요일. 점심으로 동태찌개를 먹었는데 시원해서 좋았다. 오후엔 회의 준비로 좀 바빴지만 퇴근은 제시간에 했다. 현관문을 열 때 강아지가 반겨주는 그 순간이 하루 중 가장 평화로운 ...
[2025-12-28] (평범함) 일요일. 교회에 가서 조용히 기도하고 왔다. 한 해를 무사히 보낸 것에 감사하며 차분하게 마음을 가라앉혔다. 특별한 일은 없었지만 영적으로 충만한 기분이 들어 좋았다. 저녁엔 가족...
[2026-01-26] (불쾌함) 월요일. 점심에 먹은 게 잘못됐는지 계속 속이 더부룩하다. 중요한 회의 내내 배가 아파서 집중도 못 하고 식은땀만 흘렸다. 건강 관리 잘하겠다고 해놓고 또 과식한 내 자신이 너무 ...
[2026-02-24] (불쾌함) 날씨가 갑자기 추워졌다. 출근